In [1]:
import re

In [2]:
STOPWORDS = {
    "el",
    "la",
    "los",
    "las",
    "un",
    "una",
    "unos",
    "unas",
    "de",
    "en",
    "y",
    "a",
    "que",
    "se",
    "no",
    "es",
    "son",
    "por",
    "con",
    "para",
    "del",
    "al",
    "lo",
    "le",
    "su",
}


def normalizar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúüñ\s]", "", texto)  # solo letras
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def tokenizar(texto):
    return texto.split()


def eliminar_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS]


def stemming_simple(token):
    """Stemmer muy básico para español"""
    sufijos = [
        "iendo",
        "ando",
        "iones",
        "cion",
        "mente",
        "ados",
        "adas",
        "idos",
        "idas",
        "es",
        "s",
    ]
    for suf in sufijos:
        if token.endswith(suf) and len(token) - len(suf) >= 3:
            return token[: -len(suf)]
    return token


def preprocesar(texto):
    texto = normalizar(texto)
    tokens = tokenizar(texto)
    tokens = eliminar_stopwords(tokens)
    stems = [stemming_simple(t) for t in tokens]

    return stems

In [3]:
texto = "Los estudiantes están aprendiendo programación en la universidad"

print("Original:", texto)

print("Tokens:  ", tokenizar(normalizar(texto)))

print("Sin stop:", eliminar_stopwords(tokenizar(normalizar(texto))))

print("Stems:   ", preprocesar(texto))

Original: Los estudiantes están aprendiendo programación en la universidad
Tokens:   ['los', 'estudiantes', 'están', 'aprendiendo', 'programación', 'en', 'la', 'universidad']
Sin stop: ['estudiantes', 'están', 'aprendiendo', 'programación', 'universidad']
Stems:    ['estudiant', 'están', 'aprend', 'programación', 'universidad']


## TF-IDF (Frecuencia de Término-Frecuencia Inversa de Documento) 
- Sirve para evaluar la importancia de una palabra dentro de un documento en relación con una colección de documentos (corpus). 
- Penaliza palabras muy comunes (como "el", "que") y resalta términos únicos, lo que permite clasificar textos, extraer palabras clave y mejorar motores de búsqueda.

In [4]:
import math
from collections import Counter

In [5]:
def calcular_tf(tokens):

    conteo = Counter(tokens)

    total = len(tokens)

    return {t: conteo[t] / total for t in conteo}


def calcular_idf(corpus_tokens):

    N = len(corpus_tokens)

    idf = {}

    vocab = set(t for doc in corpus_tokens for t in doc)

    for termino in vocab:
        df = sum(1 for doc in corpus_tokens if termino in doc)

        idf[termino] = math.log(N / df)

    return idf


def calcular_tfidf(corpus_raw):

    corpus_tokens = [preprocesar(doc) for doc in corpus_raw]

    idf = calcular_idf(corpus_tokens)

    resultado = []

    for tokens in corpus_tokens:
        tf = calcular_tf(tokens)

        tfidf = {t: tf[t] * idf[t] for t in tf}

        resultado.append(tfidf)

    return resultado, idf

In [6]:
corpus = [
    "el gato come pescado fresco",
    "el perro come carne roja",
    "el gato y el perro juegan juntos",
]

tfidf, idf = calcular_tfidf(corpus)

for i, doc in enumerate(tfidf):
    print(f"Doc {i + 1}:", sorted(doc.items(), key=lambda x: -x[1])[:3])

Doc 1: [('pescado', 0.27465307216702745), ('fresco', 0.27465307216702745), ('gato', 0.1013662770270411)]
Doc 2: [('carne', 0.27465307216702745), ('roja', 0.27465307216702745), ('perro', 0.1013662770270411)]
Doc 3: [('juegan', 0.27465307216702745), ('junto', 0.27465307216702745), ('gato', 0.1013662770270411)]
